# Expfit notes 0: outline

**Goal**: Fit one or multiple exponentials to (segments of) time series data, to extract time constants

**Approach**: Transform the data and use good heuristics for initial guesses, so that any optimiser will do.

## Planned functions

- [x] fit_single
- [ ] fit_double (all decaying)
- [ ] fit_triple (all decaying)
- [ ] fit_quad / fit_quadruple (all decaying)
- [ ] fit_12 (1 growing, followed by 2 decaying)
- [ ] fit_31

## Requirements on input

The method `vet_series(t, v)` checks that `t` and `v` are 1d, the same length, and t either has length 0 or is strictly increasing.

## Single exponential equation

When fitting, we use $t = a + b e^{c v}$.

This is preferred over the tau form, as for flat lines $\tau \rightarrow \infty$, which makes it a difficult parameter to fit even when the slope is moderate.

## Transformation to approximate unit square

Before fitting, a transformation is performed to:

1. Avoid extreme numbers, especially when $b$ and $c$ (or optimiser guesses of $b$ and $c$) are large.
2. Allow magic number based heuristics (although not currently used)

The time series $(t, v)$ is mapped onto an approximate unit square:

\begin{align}
x = \frac{t - t_0}{t_{n-1} - t_0} \equiv \frac{t - t_0}{r_t} &&
y = \frac{v - v_0}{v_{n-1} - v_0} \equiv \frac{v - v_0}{r_v}
\end{align}

Because $t$ is guaranteed to be strictly increasing, this places $x$ on the range $[0, 1]$
Because $v$ is assumed to be predominantly exponential, $v_{n-1}$ and $v_0$ are good approximations of its extrema, so that $y$ is _approximately_ on $[0, 1]$.

**Edge case:** If $r_v = 0$ (exactly) we set $r_v = 1$.

### Detransforming obtained parameters

Having found parameters $\tilde{a}, \tilde{b}, \tilde{c}$ in the transformed space, we have
\begin{align}
y &= \tilde{a} + \tilde{b} \exp[\tilde{c} x] \\
\frac{v - v_0}{r_v} &= \tilde{a} + \tilde{b} \exp[\tilde{c} (t - t_0) / r_t] \\
v &= v_0 + r_v \tilde{a} + r_v \tilde{b} \exp[-\tilde{c} t_0 / r_t] \exp[\tilde{c} t / r_t] \\
  &= a + b e^{c t} \\
\end{align}
for
\begin{align}
a = v_0 + r_v \tilde{a} &&
b = r_v \tilde{b} \exp[-\tilde{c} t_0 / r_t] &&
c = \tilde{c} / r_t
\end{align}
which lets us detransform parameters obtained in the transformed space.

For debugging, we can plot the known solution in the transformed space, with:
\begin{align}
\tilde{a} = \frac{a - v_0}{r_v} && 
\tilde{b} = \frac{b}{r_v} e^{c t_0} && 
\tilde{c} = c r_t
\end{align}

**Note:** The transformations for $b$ and $c$ extend to $b_i$ and $c_i$ for multiple exponentials.

## Fitting a single exponential

1. Transform to approximate unit square
2. Use a heuristic to find an initial guess $a_0, b_0, c_0$
3. Minise the MSE: $\frac{1}{n}\sum \left(y_i - a - b e^{c t_i}\right)^2$
4. Detransform the obtained parameters and return

### Fitting $c$ in log-space?

Could be better conditioned. Haven't tried.